In [4]:
!pip install torch torchvision opencv-python-headless

import os
import gc
import shutil
import random
import unittest
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import v2
import cv2

# --- 1. ROBUST PATH DISCOVERY ---
target_video_dir = None
search_candidates = [
    "./data_sample",
    "../data_sample",
    "/home/jupyter/S.E processing/data_sample",
    "/home/jupyter/data_sample"
]

for candidate in search_candidates:
    if os.path.exists(candidate):
        target_video_dir = candidate
        break

if target_video_dir is None:
    raise RuntimeError("Could not find 'data_sample' directory. Please verify folder location!")

extracted_frames_dir = "./video_frames"
if os.path.exists(extracted_frames_dir):
    shutil.rmtree(extracted_frames_dir)

def extract_keyframes(video_path, output_dir, frame_interval=15):
    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    frame_count, saved_count = 0, 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_interval == 0:
            frame_filename = os.path.join(output_dir, f"{video_name}_frame{saved_count:04d}.jpg")
            cv2.imwrite(frame_filename, frame)
            saved_count += 1
        frame_count += 1

    cap.release()
    return saved_count

real_count, fake_count = 0, 0
target_videos = []
valid_extensions = ('.mp4', '.avi', '.mov', '.mkv')

for root, dirs, files in os.walk(target_video_dir):
    for file in files:
        if file.lower().endswith(valid_extensions):
            full_path = os.path.join(root, file)
            target_videos.append(full_path)
            
            path_lower = full_path.lower()
            if 'fake' in path_lower:
                dst = os.path.join(extracted_frames_dir, 'fake')
                fake_count += extract_keyframes(full_path, dst, frame_interval=15)
            elif 'real' in path_lower or 'photos' in path_lower:
                dst = os.path.join(extracted_frames_dir, 'real')
                real_count += extract_keyframes(full_path, dst, frame_interval=15)

print(f"Target Video Path Used: '{os.path.abspath(target_video_dir)}'")
print(f"Found {len(target_videos)} total videos.")
print(f"Extracted {real_count} REAL frames and {fake_count} FAKE frames.")

# --- 2. STRATIFIED GROUP SPLIT ---
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

class_to_idx = {'fake': 0, 'real': 1}
video_to_frames = {}
real_video_ids, fake_video_ids = [], []

for category in ['fake', 'real']:
    folder = os.path.join(extracted_frames_dir, category)
    if not os.path.exists(folder):
        continue
    for fname in os.listdir(folder):
        if fname.lower().endswith(('.jpg', '.png')):
            fpath = os.path.join(folder, fname)
            video_id = fname.split('_frame')[0]
            label = class_to_idx[category]
            
            if video_id not in video_to_frames:
                video_to_frames[video_id] = []
                if category == 'real':
                    real_video_ids.append(video_id)
                else:
                    fake_video_ids.append(video_id)
            video_to_frames[video_id].append((fpath, label))

random.seed(42)
random.shuffle(real_video_ids)
random.shuffle(fake_video_ids)

def split_ids(id_list):
    n = len(id_list)
    tr = max(1, int(0.70 * n))
    va = tr + max(1, int(0.15 * n)) if n > 2 else tr
    return id_list[:tr], (id_list[tr:va] if n > 1 else id_list[:tr]), (id_list[va:] if n > 2 else id_list[tr:va])

real_train, real_val, real_test = split_ids(real_video_ids)
fake_train, fake_val, fake_test = split_ids(fake_video_ids)

train_vids = real_train + fake_train
val_vids = real_val + fake_val
test_vids = real_test + fake_test

def gather_dataset_lists(vid_list):
    paths, labels = [], []
    for vid_id in vid_list:
        for fpath, lbl in video_to_frames[vid_id]:
            paths.append(fpath)
            labels.append(lbl)
    return paths, labels

train_paths, train_labels = gather_dataset_lists(train_vids)
val_paths, val_labels = gather_dataset_lists(val_vids)
test_paths, test_labels = gather_dataset_lists(test_vids)

print(f"\nGroup Split Summary -> Train Vids: {len(train_vids)} ({len(train_paths)} frames) | "
      f"Val Vids: {len(val_vids)} ({len(val_paths)} frames) | "
      f"Test Vids: {len(test_vids)} ({len(test_paths)} frames)")

Target Video Path Used: '/home/jupyter/S.E processing/data_sample'
Found 20 total videos.
Extracted 226 REAL frames and 124 FAKE frames.

Group Split Summary -> Train Vids: 14 (246 frames) | Val Vids: 2 (27 frames) | Test Vids: 4 (77 frames)


In [5]:
# --- 3. DATASET & DATALOADERS ---
class VideoFrameDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

train_transform = v2.Compose([
    v2.Resize((128, 128)),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = v2.Compose([
    v2.Resize((128, 128)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

if len(train_paths) == 0:
    raise RuntimeError("train_paths is empty! Please run Cell 1 first.")

train_ds = VideoFrameDataset(train_paths, train_labels, train_transform)
val_ds = VideoFrameDataset(val_paths, val_labels, val_test_transform)

train_loader = DataLoader(train_ds, batch_size=max(1, min(8, len(train_ds))), shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=max(1, min(8, len(val_ds))), shuffle=False, num_workers=0)

# --- 4. CUSTOM CNN DEFINITION ---
class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        
        self.fc1 = nn.Linear(256 * 4 * 4, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = self.pool(self.relu(self.bn4(self.conv4(x))))
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)

# --- 5. TRAINING LOOP ---
print(f"Training CustomCNN on {device}...")
for epoch in range(15):
    model.train()
    running_loss, correct_train = 0.0, 0
    for inputs, lbls in train_loader:
        inputs, lbls = inputs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += (preds == lbls).sum().item()
        
    epoch_acc = (correct_train / len(train_ds)) * 100 if len(train_ds) > 0 else 0

    model.eval()
    correct_val = 0
    with torch.no_grad():
        for inputs, lbls in val_loader:
            inputs, lbls = inputs.to(device), lbls.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct_val += (preds == lbls).sum().item()
            
    val_acc = (correct_val / len(val_ds)) * 100 if len(val_ds) > 0 else 0
    print(f"Epoch {epoch+1:02d}/15 | Train Frame Acc: {epoch_acc:.2f}% | Val Frame Acc: {val_acc:.2f}%")

Training CustomCNN on cpu...
Epoch 01/15 | Train Frame Acc: 69.92% | Val Frame Acc: 100.00%
Epoch 02/15 | Train Frame Acc: 91.87% | Val Frame Acc: 55.56%
Epoch 03/15 | Train Frame Acc: 95.93% | Val Frame Acc: 100.00%
Epoch 04/15 | Train Frame Acc: 98.37% | Val Frame Acc: 59.26%
Epoch 05/15 | Train Frame Acc: 96.34% | Val Frame Acc: 55.56%
Epoch 06/15 | Train Frame Acc: 97.97% | Val Frame Acc: 66.67%
Epoch 07/15 | Train Frame Acc: 99.19% | Val Frame Acc: 48.15%
Epoch 08/15 | Train Frame Acc: 98.37% | Val Frame Acc: 100.00%
Epoch 09/15 | Train Frame Acc: 99.19% | Val Frame Acc: 81.48%
Epoch 10/15 | Train Frame Acc: 98.37% | Val Frame Acc: 100.00%
Epoch 11/15 | Train Frame Acc: 99.19% | Val Frame Acc: 77.78%
Epoch 12/15 | Train Frame Acc: 99.59% | Val Frame Acc: 48.15%
Epoch 13/15 | Train Frame Acc: 99.19% | Val Frame Acc: 66.67%
Epoch 14/15 | Train Frame Acc: 100.00% | Val Frame Acc: 81.48%
Epoch 15/15 | Train Frame Acc: 99.59% | Val Frame Acc: 100.00%


In [6]:
# --- UNIT TEST: FILE FORMAT VALIDATION ---
print("="*50)
print("🧪 UNIT TEST: FILE FORMAT VALIDATION")
print("="*50)

ALLOWED_EXTENSIONS = {'.mp4', '.avi'}

def is_valid_video_format(filename):
    if not isinstance(filename, str) or not filename.strip():
        return False
    ext = os.path.splitext(filename)[1].lower()
    return ext in ALLOWED_EXTENSIONS

class TestVideoFormatValidation(unittest.TestCase):
    def test_allowed_formats_pass(self):
        self.assertTrue(is_valid_video_format("sample_video.mp4"))
        self.assertTrue(is_valid_video_format("clip.avi"))
        self.assertTrue(is_valid_video_format("UPPERCASE_TEST.MP4"))

    def test_unsupported_formats_fail(self):
        self.assertFalse(is_valid_video_format("document.pdf"))
        self.assertFalse(is_valid_video_format("script.py"))
        self.assertFalse(is_valid_video_format("picture.png"))
        self.assertFalse(is_valid_video_format("unsupported.mkv"))

    def test_edge_cases_fail(self):
        self.assertFalse(is_valid_video_format("no_extension"))
        self.assertFalse(is_valid_video_format(""))

suite = unittest.TestLoader().loadTestsFromTestCase(TestVideoFormatValidation)
unittest.TextTestRunner(verbosity=2).run(suite)

test_allowed_formats_pass (__main__.TestVideoFormatValidation) ... ok
test_edge_cases_fail (__main__.TestVideoFormatValidation) ... ok
test_unsupported_formats_fail (__main__.TestVideoFormatValidation) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.004s

OK


🧪 UNIT TEST: FILE FORMAT VALIDATION


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

In [ ]:
# --- INTEGRATION TEST & METRICS ---
print("="*60)
print("🚀 INTEGRATION TEST: END-TO-END PIPELINE & METRICS")
print("="*60)

test_video_paths = [v for v in target_videos if os.path.splitext(os.path.basename(v))[0] in test_vids]

y_true = []
y_pred = []

for vid_path in test_video_paths:
    actual = 0 if 'fake' in vid_path.lower() else 1
    
    cap = cv2.VideoCapture(vid_path)
    if not cap.isOpened():
        continue
        
    frame_preds = []
    frame_count = 0
    model.eval()
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % 15 == 0:
                img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                tensor = val_test_transform(img).unsqueeze(0).to(device)
                prob = F.softmax(model(tensor), dim=1)[0][1].item()
                frame_preds.append(prob)
            frame_count += 1
    cap.release()
    
    if frame_preds:
        avg_real_prob = np.mean(frame_preds)
        avg_fake_prob = 1.0 - avg_real_prob
        pred = 1 if avg_real_prob > 0.5 else 0
        
        y_true.append(actual)
        y_pred.append(pred)
        
        verdict = "REAL" if pred == 1 else "FAKE"
        actual_str = "REAL" if actual == 1 else "FAKE"
        confidence = max(avg_real_prob, avg_fake_prob) * 100
        is_correct = "✅" if pred == actual else "❌"
        print(f"[{is_correct}] Video: {os.path.basename(vid_path):<30} | Pred: {verdict:<4} ({confidence:.1f}%) | Actual: {actual_str}")

y_true = np.array(y_true)
y_pred = np.array(y_pred)

tp = np.sum((y_true == 1) & (y_pred == 1))
fp = np.sum((y_true == 0) & (y_pred == 1))
fn = np.sum((y_true == 1) & (y_pred == 0))
tn = np.sum((y_true == 0) & (y_pred == 0))

accuracy = (tp + tn) / len(y_true) if len(y_true) > 0 else 0.0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("\n" + "="*50)
print("📊 FINAL CLASSIFICATION METRICS")
print("="*50)
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1_score:.4f}")
print("="*50)